# PlugIR · Remaining S2 and S3 variants

All methods use the same 200 dialogue starts, Qwen and BLIP checkpoints, and 50,000-image index. The latest query ranks the user-facing results. Each dialogue stops at its first top-10 hit or after 10 questions.

- **Current MMR** selects ten captions from the latest-query pool using current-query relevance and a BLIP cosine redundancy penalty.
- **Split 5+5** selects five captions from the top of that pool and five distinct captions ranked by earlier searches. With no earlier search, it uses the current ranking.
- **S2 Original + Current** fuses the original and latest ranks for the candidate pool and keeps PlugIR’s k-means and representative rule.

The comparison includes saved vanilla, S2 All-history, and the three completed S3 runs. The new variants alternate in batches using one GPU worker.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

# Start Jupyter in the directory containing this notebook, or set PACKAGE explicitly.
PACKAGE = Path.cwd().resolve()
RESULTS = PACKAGE / "results"
REPORT = PACKAGE / "source" / "report_hits.py"

# Change these paths if your checkout or MLX environment lives elsewhere.
PROJECT = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / "ChatIR-Cornell2026").is_dir()),
    PACKAGE.parent,
)
REPO = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / ".git").exists()),
    PROJECT / "ChatIR-Cornell2026",
)
BLIP_PYTHON = REPO / ".venv" / "bin" / "python"
MLX_ENV = PROJECT / "tmp" / "agentir-mlx-vlm-env"
QWEN_REVISION = "8b2b98c00a6b4d291155e4890773ca8f769aee53"
SNAPSHOT = (REPO / "cache/huggingface/hub/models--mlx-community--Qwen3.5-9B-4bit"
            / "snapshots" / QWEN_REVISION)
ARMS = ["s3_current_mmr", "s3_split_5_5", "s2_original_current_pool"]
REFERENCE_ARMS = ["vanilla", "s2_history_pool", "s3_fused_top10", "s3_current_top10", "s3_history_mmr"]
REPORT_ARMS = [*REFERENCE_ARMS, *ARMS]
SAMPLE = json.loads((PACKAGE / "sample.json").read_text())
DIALOGUE_COUNT = len(SAMPLE["tasks"])
print(f"{DIALOGUE_COUNT} dialogues per method, seed {SAMPLE['seed']}")

## Check saved progress

The matched count includes dialogues finished in every method.

In [ ]:
def saved_ids(arm):
    folder = (RESULTS / "new_runs" if arm in ARMS else RESULTS) / arm
    return {int(path.stem) for path in (folder / "dialogues").glob("*.json")}

for arm in REPORT_ARMS:
    print(f"{arm}: {len(saved_ids(arm))}/{DIALOGUE_COUNT}")
print(f"Matched: {len(set.intersection(*(saved_ids(arm) for arm in REPORT_ARMS)))}/{DIALOGUE_COUNT}")

## Run or resume on Apple Silicon

Use the progress cell while a run is active. Choose a new results directory for a separate replication.

In [ ]:
from shutil import copy2, copytree
from tempfile import TemporaryDirectory

for dependency in (BLIP_PYTHON, MLX_ENV / "bin/python", SNAPSHOT / "config.json"):
    if not dependency.exists():
        raise FileNotFoundError(f"{dependency} is missing. See README.md for setup.")

reference = PACKAGE / "results"
if RESULTS.resolve() != reference.resolve():
    for arm in REFERENCE_ARMS:
        copytree(reference / arm, RESULTS / arm, dirs_exist_ok=True)

# Stage the recorded source in the layout used by the original runner.
with TemporaryDirectory(prefix="agentir-s1-") as stage:
    workspace = Path(stage)
    bundle_dir = workspace / "deliverables/full_dialogue_colab"
    source_dir = bundle_dir / "source"
    source_dir.mkdir(parents=True)
    (workspace / "ChatIR-Cornell2026").symlink_to(REPO, target_is_directory=True)
    (workspace / "tmp").mkdir()
    (workspace / "tmp/agentir-mlx-vlm-env").symlink_to(MLX_ENV, target_is_directory=True)
    for source in (PACKAGE / "source").glob("*.py"):
        copy2(source, source_dir / source.name)
    archive = "agentir_full_dialogue_bundle_v2.zip"
    (bundle_dir / archive).symlink_to(PACKAGE / archive)
    subprocess.run(
        ["caffeinate", "-i", str(BLIP_PYTHON), "-u", str(source_dir / "local_hits.py"),
         "--snapshot", str(SNAPSHOT), "--output", str(RESULTS / "new_runs"), "--limit", str(DIALOGUE_COUNT),
         "--arms", *ARMS],
        check=True,
    )

## Plot the matched comparison

The default requires all 200 dialogues across all methods. Set `ALLOW_PARTIAL = True` to inspect an interim curve labeled with its smaller sample size.

In [ ]:
from IPython.display import Image, display

ALLOW_PARTIAL = False
command = [sys.executable, str(REPORT), str(RESULTS)]
if ALLOW_PARTIAL:
    command.append("--allow-partial")
figures = subprocess.check_output(command, text=True).strip()
display(Image(filename=str(Path(figures) / "hits_at_10.png")))